# Imports

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %pip install scikit-learn


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.sparse import coo_matrix
import random
import numpy as np
import pickle
import time
from scipy.special import expit
from pathlib import Path
import os

# Load Item Data

In [4]:
# import os
# os.listdir("/content/drive")

In [5]:
os.getcwd()

'/noamk102/user/yagelalfasi/DL2_Project/Amazon'

# Load data

## User–Item interaction matrix

This matrix stores explicit ratings
Shape ∈R∣U∣×∣I∣

In [6]:
items_fashion_cleaned_df = pd.read_csv("Amazon_items_cleaned.csv")
ratings_cleaned_df = pd.read_csv( "Amazon_ratings_cleaned.csv")

In [7]:
ratings_cleaned_df.shape

(8528231, 5)

In [8]:
# ratings_cleaned_df.head(10)

In [9]:
user_ids = ratings_cleaned_df["user_id"].unique()
item_ids = ratings_cleaned_df["parent_asin"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {i: j for j, i in enumerate(item_ids)}

idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {j: i for i, j in item_to_idx.items()}

## Load the coo_matrix

In [10]:
from scipy.sparse import load_npz
user_item_matrix = load_npz( "user_item_matrix.npz")

In [11]:
coo = user_item_matrix.tocoo()
list(zip(coo.row[:10], coo.col[:10], coo.data[:10]))


[(np.int32(0), np.int32(0), np.float32(1.0)),
 (np.int32(0), np.int32(1), np.float32(1.0)),
 (np.int32(0), np.int32(2), np.float32(1.0)),
 (np.int32(0), np.int32(3), np.float32(1.0)),
 (np.int32(0), np.int32(4), np.float32(1.0)),
 (np.int32(1), np.int32(5), np.float32(1.0)),
 (np.int32(1), np.int32(6), np.float32(1.0)),
 (np.int32(1), np.int32(7), np.float32(1.0)),
 (np.int32(1), np.int32(8), np.float32(1.0)),
 (np.int32(1), np.int32(9), np.float32(1.0))]

In [12]:
len(user_ids)

1089923

In [13]:
user_item_matrix.shape

(1089923, 182207)

In [14]:
print("user item interation matrix shape:", user_item_matrix.shape)
print("user item interation matrix nnz (ratings):", user_item_matrix.nnz)

assert user_item_matrix.shape[0] == len(user_ids)
assert user_item_matrix.shape[1] == len(item_ids)
assert user_item_matrix.nnz == len(ratings_cleaned_df)

user item interation matrix shape: (1089923, 182207)
user item interation matrix nnz (ratings): 8528231


In [15]:
items_fashion_cleaned_df.shape

(182207, 17)

## Item–Category information matrix


Items may belong to multiple categories → multiple 1's per row.

Shape∈{0,1}∣I∣×∣C∣

### Create category index mapping

In [16]:
items_fashion_cleaned_df["categories_list_pruned"]

0         ['Clothing, Shoes & Jewelry', 'Women', 'Clothi...
1         ['Clothing, Shoes & Jewelry', 'Novelty & More'...
2         ['Clothing, Shoes & Jewelry', 'Women', 'Clothi...
3         ['Clothing, Shoes & Jewelry', 'Women', 'Clothi...
4         ['Clothing, Shoes & Jewelry', 'Women', 'Shoes'...
                                ...                        
182202    ['Clothing, Shoes & Jewelry', 'Women', 'Jewelr...
182203    ['Clothing, Shoes & Jewelry', 'Women', 'Clothi...
182204    ['Clothing, Shoes & Jewelry', 'Women', 'Jewelr...
182205    ['Clothing, Shoes & Jewelry', 'Men', 'Accessor...
182206    ['Clothing, Shoes & Jewelry', 'Men', 'Shoes', ...
Name: categories_list_pruned, Length: 182207, dtype: object

In [17]:
# All retained categories
all_categories = sorted({
    cat
    for cats in items_fashion_cleaned_df["categories_list_pruned"]
    for cat in cats
})

category_to_idx = {c: i for i, c in enumerate(all_categories)}
idx_to_category = {i: c for c, i in category_to_idx.items()}

In [18]:
# items_fashion_cleaned_df.head(5)

In [19]:
print(len(all_categories))
print(len(category_to_idx))
# type(idx_to_category)

65
65


### Build Item × Category matrix (multi-category safe)

In [20]:
cat_rows = []
cat_cols = []

for _, row in items_fashion_cleaned_df.iterrows():
    item_id = row["parent_asin"]

    # Skip items not in ratings matrix
    if item_id not in item_to_idx:
        continue

    item_idx = item_to_idx[item_id]

    for cat in row["categories_list_pruned"]:
        cat_rows.append(item_idx)
        cat_cols.append(category_to_idx[cat])

item_category_matrix = coo_matrix(
    (np.ones(len(cat_rows)), (cat_rows, cat_cols)),
    shape=(len(item_to_idx), len(category_to_idx))
)

### Sanity Checks

In [21]:
print("item category information matrix shape:", item_category_matrix.shape)
print("item category information matrix nnz (item–category links):", item_category_matrix.nnz)

assert item_category_matrix.shape[0] == user_item_matrix.shape[1]     # same number of items
assert item_category_matrix.nnz >= len(items_fashion_cleaned_df)

item category information matrix shape: (182207, 65)
item category information matrix nnz (item–category links): 14640551


## Cross Matrix Alignment Checks

### Item alignment

In [22]:
# All items used in ratings
items_in_user_item_matrix = set(item_ids)

# All items used in categories
items_in_item_category_matrix = set(items_fashion_cleaned_df["parent_asin"])

assert items_in_user_item_matrix.issubset(items_in_item_category_matrix), \
    "Some items in user item matrix have no category metadata!"

### Index consistency

In [23]:
random_item = random.choice(list(items_in_user_item_matrix))

item_idx = item_to_idx[random_item]

print("Item ASIN:", random_item)
print("Categories:",
      items_fashion_cleaned_df
      .loc[items_fashion_cleaned_df["parent_asin"] == random_item,
           "categories_list_pruned"]
      .values[0])

print("Nonzero category indices:",
      item_category_matrix.getrow(item_idx).nonzero()[1])

print("Category names:",
      [idx_to_category[i] for i in item_category_matrix.getrow(item_idx).nonzero()[1]])

Item ASIN: B01C645RKA
Categories: ['Clothing, Shoes & Jewelry', 'Men', 'Shoes', 'Oxfords']
Nonzero category indices: [63 62 61 58 57 56 53 52 50 47 46 45 44 43 42 38 37 31 27 25 22 15  5  4
  3  0]
Category names: ['y', 'x', 'w', 't', 's', 'r', 'o', 'n', 'l', 'i', 'h', 'g', 'f', 'e', 'd', ']', '[', 'S', 'O', 'M', 'J', 'C', ',', "'", '&', ' ']


The printed categories match so the pipeline is correct.

### Check Density Sanity

In [24]:
print("Avg ratings per user:", user_item_matrix.nnz / user_item_matrix.shape[0])
print("Avg ratings per item:", user_item_matrix.nnz / user_item_matrix.shape[1])
print("Avg categories per item:", item_category_matrix.nnz / item_category_matrix.shape[0])

Avg ratings per user: 7.824617885850651
Avg ratings per item: 46.80517762764328
Avg categories per item: 80.35119945995488


In [25]:
from pathlib import Path


np.save("user_to_idx.npy", user_to_idx)
np.save("item_to_idx.npy", item_to_idx)


## To Load Later

In [26]:
# user_to_idx = np.load(BASE / "user_to_idx.npy", allow_pickle=True).item()
# item_to_idx = np.load(BASE / "item_to_idx.npy", allow_pickle=True).item()

## For now to delete from RAM

In [27]:
del user_to_idx, item_to_idx
import gc; gc.collect()

40

# Run MF

Create positive and negative indexes in the user-item interaction matrix for future use in the MF

In [28]:
# num_users_mf = user_item_matrix.shape[0]
# num_items_mf = user_item_matrix.shape[1]

# pos_ex = {}
# neg_ex = {}
# pos_ex_num = {}

# for user_idx in range(num_users_mf):
#     # Get the sparse row for the current user
#     user_row = user_item_matrix.getrow(user_idx)

#     # Get column indices of positive interactions (non-zero values)
#     positive_item_indices = user_row.nonzero()[1].tolist()
#     pos_ex[user_idx] = positive_item_indices
#     pos_ex_num[user_idx] = len(positive_item_indices)

#     # Get all possible item indhgbices
#     all_item_indices = set(range(num_items_mf))
#     # Get item indices of negative interactions
#     negative_item_indices = list(all_item_indices - set(positive_item_indices))
#     neg_ex[user_idx] = negative_item_indices

In [29]:
user_item_matrix.shape

(1089923, 182207)

# Trial (new code for MF)
Preparing data smaples

Positive sampling

In [30]:
import numpy as np
from scipy.special import expit

# CSR matrix
R = user_item_matrix

users_pos, items_pos = R.nonzero()
num_users, num_items = R.shape

Negative sampling

In [31]:
def sample_negatives(R, num_neg=1):
    """
    For each positive interaction, sample num_neg negative items.
    """
    user_neg = []
    item_neg = []

    num_items = R.shape[1]

    users_pos = np.unique(R.nonzero()[0])

    for u in users_pos:
        pos_items = set(R[u].indices)

        cnt = 0
        target = num_neg * len(pos_items)

        while cnt < target:
            j = np.random.randint(num_items)
            if j not in pos_items:
                user_neg.append(u)
                item_neg.append(j)
                cnt += 1

    return np.array(user_neg), np.array(item_neg)


# MF class

In [32]:
import numpy as np
from scipy.special import expit

class SparseMF:
    def __init__(self, num_users, num_items, K=20, lr=0.01, reg=0.01):
        self.K = K
        self.lr = lr
        self.reg = reg

        self.P = np.random.normal(0, 0.1, (num_users, K))
        self.Q = np.random.normal(0, 0.1, (num_items, K))
        self.b_u = np.zeros(num_users)
        self.b_i = np.zeros(num_items)

    def train(self, users, items, labels, epochs=10, batch_size=1024):
        n = len(users)
    
        for epoch in range(epochs):
            perm = np.random.permutation(n)
            users_shuf = users[perm]
            items_shuf = items[perm]
            labels_shuf = labels[perm]
    
            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                u = users_shuf[start:end]
                i = items_shuf[start:end]
                y = labels_shuf[start:end]
    
                Pu = self.P[u].copy()
                Qi = self.Q[i].copy()
    
                score = self.b_u[u] + self.b_i[i] + np.sum(Pu * Qi, axis=1)
                pred = expit(score)
    
                err = y - pred
    
                grad_bu = err - self.reg * self.b_u[u]
                grad_bi = err - self.reg * self.b_i[i]
                grad_P  = err[:, None] * Qi - self.reg * Pu
                grad_Q  = err[:, None] * Pu - self.reg * Qi
    
                np.add.at(self.b_u, u, self.lr * grad_bu)
                np.add.at(self.b_i, i, self.lr * grad_bi)
                np.add.at(self.P,   u, self.lr * grad_P)
                np.add.at(self.Q,   i, self.lr * grad_Q)

    
            print(f"Epoch {epoch+1} done")


    # 🔹 RAW score (for AUC & ranking)
    def predict(self, users, items):
        return (
            self.b_u[users] +
            self.b_i[items] +
            np.sum(self.P[users] * self.Q[items], axis=1)
        )

    # 🔹 Full-item prediction for ONE user (for top-K)
    def predict_user(self, user_id):
        return (
            self.b_u[user_id] +
            self.b_i +
            self.P[user_id] @ self.Q.T
        )


# Training Data

In [33]:
# Positive samples
users_p, items_p = R.nonzero()
labels_p = np.ones(len(users_p))

# Negative samples
users_n, items_n = sample_negatives(R, num_neg=1)
labels_n = np.zeros(len(users_n))

# Combine
users = np.concatenate([users_p, users_n])
items = np.concatenate([items_p, items_n])
labels = np.concatenate([labels_p, labels_n])

In [34]:
# =========================================
# CLEAN DATA PIPELINE FOR IMPLICIT MF
# =========================================

import numpy as np
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from collections import defaultdict

# -----------------------------------------
# Step 1: Get POSITIVES ONLY from R
# -----------------------------------------
users_pos, items_pos = R.nonzero()

# -----------------------------------------
# Step 2: Select 400000 users (from positives)
# -----------------------------------------
np.random.seed(42)

all_users = np.unique(users_pos)
selected_users = np.random.choice(all_users, size=400000, replace=False)

mask = np.isin(users_pos, selected_users)

users_pos_sub = users_pos[mask]
items_pos_sub = items_pos[mask]

print("Using positives:", len(users_pos_sub))
print("Using users:", len(np.unique(users_pos_sub)))

# -----------------------------------------
# Step 3: Remap users to 0..N-1 (keep item ids)
# -----------------------------------------
sub_users_unique = np.unique(users_pos_sub)
user2id = {u: i for i, u in enumerate(sub_users_unique)}

users_pos_sub_ids = np.array([user2id[u] for u in users_pos_sub])
items_pos_sub_ids = items_pos_sub

num_users = len(sub_users_unique)
num_items = R.shape[1]

print("num_users =", num_users)
print("num_items =", num_items)

# -----------------------------------------
# Step 4: Split POSITIVES ONLY into train / val
# -----------------------------------------
users_tr_pos, users_val_pos, items_tr_pos, items_val_pos = train_test_split(
    users_pos_sub_ids,
    items_pos_sub_ids,
    test_size=0.2,
    random_state=42
)

print("Train positives:", len(users_tr_pos))
print("Val positives:", len(users_val_pos))

# -----------------------------------------
# Step 5: Negative sampling function
# -----------------------------------------
def sample_negatives_for_pairs(users_pos, items_pos, R, num_neg=1):
    num_items = R.shape[1]

    user_neg = []
    item_neg = []

    pos_by_user = defaultdict(set)
    for u, i in zip(users_pos, items_pos):
        pos_by_user[u].add(i)

    for u, pos_items in pos_by_user.items():
        target = num_neg * len(pos_items)
        cnt = 0
        while cnt < target:
            j = np.random.randint(num_items)
            if j not in pos_items:
                user_neg.append(u)
                item_neg.append(j)
                cnt += 1

    return np.array(user_neg), np.array(item_neg)

# -----------------------------------------
# Step 6: Sample negatives SEPARATELY
# -----------------------------------------
users_tr_neg, items_tr_neg = sample_negatives_for_pairs(
    users_tr_pos, items_tr_pos, R, num_neg=1
)

users_val_neg, items_val_neg = sample_negatives_for_pairs(
    users_val_pos, items_val_pos, R, num_neg=1
)

print("Train negatives:", len(users_tr_neg))
print("Val negatives:", len(users_val_neg))

# -----------------------------------------
# Step 7: Build final train / val sets
# -----------------------------------------
users_tr = np.concatenate([users_tr_pos, users_tr_neg])
items_tr = np.concatenate([items_tr_pos, items_tr_neg])
y_tr = np.concatenate([np.ones(len(users_tr_pos)), np.zeros(len(users_tr_neg))])

users_val = np.concatenate([users_val_pos, users_val_neg])
items_val = np.concatenate([items_val_pos, items_val_neg])
y_val = np.concatenate([np.ones(len(users_val_pos)), np.zeros(len(users_val_neg))])

# Shuffle
perm = np.random.permutation(len(users_tr))
users_tr, items_tr, y_tr = users_tr[perm], items_tr[perm], y_tr[perm]

perm = np.random.permutation(len(users_val))
users_val, items_val, y_val = users_val[perm], items_val[perm], y_val[perm]

print("Final train size:", len(users_tr))
print("Final val size:", len(users_val))

print("y_tr mean:", float(y_tr.mean()))
print("y_val mean:", float(y_val.mean()))

# -----------------------------------------
# Step 8: Build R_train and R_val (for Top-K eval)
# -----------------------------------------
R_train = csr_matrix(
    (np.ones(len(users_tr_pos)), (users_tr_pos, items_tr_pos)),
    shape=(num_users, num_items)
)

R_val = csr_matrix(
    (np.ones(len(users_val_pos)), (users_val_pos, items_val_pos)),
    shape=(num_users, num_items)
)

print("R_train shape:", R_train.shape)
print("R_val shape:", R_val.shape)


Using positives: 3133692
Using users: 400000
num_users = 400000
num_items = 182207
Train positives: 2506953
Val positives: 626739
Train negatives: 2506953
Val negatives: 626739
Final train size: 5013906
Final val size: 1253478
y_tr mean: 0.5
y_val mean: 0.5
R_train shape: (400000, 182207)
R_val shape: (400000, 182207)


In [35]:
print("Pos samples:", len(users_p))
print("Neg samples:", len(users_n))


Pos samples: 8528231
Neg samples: 8528231


In [36]:
print("y_tr mean:", float(y_tr.mean()))
print("y_val mean:", float(y_val.mean()))


y_tr mean: 0.5
y_val mean: 0.5


In [37]:
from sklearn.metrics import roc_auc_score

mf = SparseMF(num_users, num_items, K=20, lr=0.005, reg=0.0001)

scores0 = mf.predict(users_val, items_val)
print("AUC before:", roc_auc_score(y_val, scores0))

mf.train(users_tr, items_tr, y_tr, epochs=20)

scores1 = mf.predict(users_val, items_val)
print("AUC after:", roc_auc_score(y_val, scores1))


AUC before: 0.5000815359110117
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done
Epoch 11 done
Epoch 12 done
Epoch 13 done
Epoch 14 done
Epoch 15 done
Epoch 16 done
Epoch 17 done
Epoch 18 done
Epoch 19 done
Epoch 20 done
AUC after: 0.7782307881858855


In [38]:
print("OK")

OK


# Training!

In [ ]:
print("=== TRAINING START CHECK ===")
print(f"Users range: [{users_tr.min()}, {users_tr.max()}]")
print(f"Items range: [{items_tr.min()}, {items_tr.max()}]")
print(f"Labels mean (pos ratio): {labels.mean():.4f}")
print(f"Num samples: {len(labels)}")
print(f"Embedding users: {num_users}, items: {num_items}")

In [ ]:
num_users_tr = num_users          # from the split cell (should be 4000)
num_items_tr = num_items 

In [ ]:
mf = SparseMF(num_users_tr, num_items_tr, K=20, lr=0.005, reg=0.01)
mf.train(users_tr, items_tr, y_tr, epochs=10)

In [ ]:
import pickle

# Save the trained model to a file
with open('sparsemf_model.pkl', 'wb') as file:
    pickle.dump(mf, file)
    
print("Model saved successfully!")

In [ ]:
# def l2_norm(x):
#     return float(np.sqrt((x * x).sum()))

# P0 = mf.P.copy()
# Q0 = mf.Q.copy()
# bu0 = mf.b_u.copy()
# bi0 = mf.b_i.copy()

# mf.train(users_tr, items_tr, y_tr, epochs=1)

# print("dP:", l2_norm(mf.P - P0))
# print("dQ:", l2_norm(mf.Q - Q0))
# print("dbu:", l2_norm(mf.b_u - bu0))
# print("dbi:", l2_norm(mf.b_i - bi0))


# Recommendations

In [ ]:
print("Model num_items:", num_items_tr)
print("R_val num_items:", R_val.shape[1])


In [ ]:
def recommend(mf, user_idx, top_k=10, seen=None):
    scores = mf.b_u[user_idx] + mf.b_i + mf.P[user_idx] @ mf.Q.T
    if seen is not None:
        scores[list(seen)] = -np.inf
    return np.argsort(scores)[-top_k:][::-1]

In [ ]:
u = 0
seen_items = set(R_train[u].indices)   # ONLY training items
top_items = recommend(mf, u, top_k=10, seen=seen_items)

In [ ]:
top_items

# Evaluation

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import log_loss


scores = mf.predict(users_val, items_val)
auc = roc_auc_score(y_val, scores)
print(f"AUC: {auc:.4f}")



probs = 1 / (1 + np.exp(-scores))  # sigmoid if needed
ll = log_loss(y_val, probs)
print(f"Log loss: {ll:.4f}")


pred = (scores > 0).astype(int)
acc = (pred == y_val).mean()
print(f"Accuracy: {acc:.4f}")

Ranking evaluation

In [ ]:
## Create R_val

from scipy.sparse import coo_matrix

# Keep only POSITIVE interactions
mask = y_val == 1

R_val = coo_matrix(
    (np.ones(mask.sum()), (users_val[mask], items_val[mask])),
    shape=(num_users, num_items)
).tocsr()


In [ ]:
def recommend_top_k(mf, R, user_id, K=10):
    seen_items = set(R[user_id].indices)
    scores = mf.predict_user(user_id)

    scores[list(seen_items)] = -np.inf
    return np.argsort(scores)[-K:][::-1]

In [ ]:
import os
import numpy as np
import pickle

def precision_recall_at_k(mf, R_val, users, K=10, save_every=10_000, out_dir="pr_checkpoints"):
    os.makedirs(out_dir, exist_ok=True)

    precisions, recalls = [], []

    job_id = os.environ.get("SLURM_JOB_ID", "no_job_id")
    log_file = f"logging_{job_id}.txt"

    with open(log_file, "a") as f:
        f.write(f"JOB_ID={job_id}\n")

    total_users = len(users)

    for idx, u in enumerate(users, start=1):
        true_items = set(R_val[u].indices)
        if idx == 1 or idx % save_every == 0:
            with open(log_file, "a") as f:
                f.write(f"logged user #{idx}/ {total_users}\n")

        if len(true_items) == 0:
            continue

        recs = set(recommend_top_k(mf, R_train, u, K))
        hits = len(recs & true_items)

        # print("val items:", R_val[u].indices[:10])
        # print("recs:", list(recs))
        # print("hits:", set(recs) & set(R_val[u].indices))



        precisions.append(hits / K)
        recalls.append(hits / len(true_items))

        # Save snapshot every save_every users
        if idx % save_every == 0:
            p_now = float(np.mean(precisions))
            r_now = float(np.mean(recalls))

            ckpt_file = os.path.join(out_dir, f"pr_at_{idx}.pkl")
            with open(ckpt_file, "wb") as f:
                pickle.dump({
                    "num_users": idx,
                    "precision": p_now,
                    "recall": r_now
                }, f)

            print(f"Saved snapshot at {idx}: P={p_now:.4f}, R={r_now:.4f}")

    # Final snapshot
    p_final = float(np.mean(precisions))
    r_final = float(np.mean(recalls))

    ckpt_file = os.path.join(out_dir, f"pr_at_{total_users}.pkl")
    with open(ckpt_file, "wb") as f:
        pickle.dump({
            "num_users": total_users,
            "precision": p_final,
            "recall": r_final
        }, f)

    return p_final, r_final


In [ ]:
users_eval = np.unique(users_val)
p, r = precision_recall_at_k(mf, R_val, users_eval, K=10)
print(f"Precision@10: {p:.4f}")
print(f"Recall@10: {r:.4f}")

NDCG@K

In [ ]:
# import numpy as np

# def ndcg_at_k(mf, R_val, users, K=10):
#     ndcgs = []

#     for u in users:
#         true_items = set(R_val[u].indices)
#         if not true_items:
#             continue

#         recs = recommend_top_k(mf, R_val, u, K)

#         dcg = 0.0
#         for i, item in enumerate(recs):
#             if item in true_items:
#                 dcg += 1 / np.log2(i + 2)

#         idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_items), K)))
#         ndcgs.append(dcg / idcg if idcg > 0 else 0)

#     return np.mean(ndcgs)

In [ ]:
# ndcg = ndcg_at_k(mf, R_val, users_eval, K=10)
# print(f"NDCG@10: {ndcg:.4f}")

In [ ]:
print("OK")

## Matryoshka Sparse Autoencoder (SAE) - New Cells

In [34]:
# Sampling: 100-row mini version of the data for quick sanity checks
import numpy as np
import torch
from torch.utils.data import DataLoader

# Sample 100 rows from the original ratings dataframe
df_sample = ratings_cleaned_df.sample(n=100, random_state=42).reset_index(drop=True)

# Build local int-index mappings for the sample
sample_user_list = list(df_sample["user_id"].unique())
sample_item_list = list(df_sample["parent_asin"].unique())
su2i = {u: i for i, u in enumerate(sample_user_list)}
si2i = {it: j for j, it in enumerate(sample_item_list)}

n_users_s = len(sample_user_list)
n_items_s = len(sample_item_list)

# Train a tiny MF on the sample (use the existing SparseMF class)
users_s = np.array([su2i[u] for u in df_sample["user_id"]])
items_s = np.array([si2i[i] for i in df_sample["parent_asin"]])
labels_s = np.ones(len(users_s))
mf_sample = SparseMF(n_users_s, n_items_s, K=20, lr=0.01, reg=0.01)
mf_sample.train(users_s, items_s, labels_s, epochs=3)

# Build the mf_recommender object the SAE cells expect
mf_recommender = mf_sample

# Build tensors + DataLoaders
user_tensor = torch.tensor(mf_recommender.P, dtype=torch.float32)
item_tensor = torch.tensor(mf_recommender.Q, dtype=torch.float32)
dataset_users = user_tensor
dataset_items = item_tensor
user_loader = DataLoader(dataset_users, batch_size=32, shuffle=True)
item_loader = DataLoader(dataset_items, batch_size=32, shuffle=True)

print(f"Sample df rows     : {len(df_sample)}")
print(f"Unique sample users: {n_users_s}")
print(f"Unique sample items: {n_items_s}")
print(f"user_tensor shape  : {user_tensor.shape}")
print(f"item_tensor shape  : {item_tensor.shape}")

Epoch 1 done
Epoch 2 done
Epoch 3 done
Sample df rows     : 100
Unique sample users: 100
Unique sample items: 100
user_tensor shape  : torch.Size([100, 20])
item_tensor shape  : torch.Size([100, 20])


In [35]:
print('K' in dir(), 'num_users' in dir(), 'num_items' in dir())

False True True


In [36]:
print(num_users, num_items)

1089923 182207


In [37]:
print([k for k in dir() if 'auto' in k.lower() or 'sae' in k.lower() or 'encoder' in k.lower()])

[]


In [46]:
from typing import Callable, Any
interaction_embeddings = dataset_users #[:, 2:]

In [47]:
def LN(x: torch.Tensor, eps: float = 1e-5) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if type(x) == np.ndarray:
      x= torch.from_numpy(x)
    mu = x.mean(dim=-1, keepdim=True)
    x = x - mu
    std = x.std(dim=-1, keepdim=True)
    x = x / (std + eps)
    return x, mu, std


class Autoencoder(nn.Module):

    def __init__(
        self, n_latents: int, n_inputs: int, activation: Callable = nn.ReLU(), tied: bool = False,
        normalize: bool = False
    ) -> None:
        super().__init__()

        self.pre_bias = nn.Parameter(torch.zeros(n_inputs))
        self.encoder: nn.Module = nn.Linear(n_inputs, n_latents, bias=False)
        self.latent_bias = nn.Parameter(torch.zeros(n_latents))
        self.activation = activation
        if tied:
            self.decoder: nn.Linear | TiedTranspose = TiedTranspose(self.encoder)
        else:
            self.decoder = nn.Linear(n_latents, n_inputs, bias=False)
        self.normalize = normalize
        self.loss = []
        self.test_subset_users_ind = test_subset_users ####
        self.test_subset_items_ind = test_subset_items ####
        self.weights_loss = [1,0.0001,0,0] ####
        self.test = test_flag


    def encode_pre_act(self, x: torch.Tensor, latent_slice: slice = slice(None)) -> torch.Tensor:
        if type(x) == np.ndarray:
          x= torch.from_numpy(x)
        x = pad_or_truncate_tensor(x, self.pre_bias.shape[0])
        x = x - self.pre_bias
        latents_pre_act = F.linear(
            x, self.encoder.weight[latent_slice], self.latent_bias[latent_slice]
        )
        return latents_pre_act

    def preprocess(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, Any]]:
        if not self.normalize:
            return x, dict()
        x, mu, std = LN(x)
        return x, dict(mu=mu, std=std)

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, Any]]:
        x, info = self.preprocess(x)
        return self.activation(self.encode_pre_act(x)), info

    def decode(self, latents: torch.Tensor, info: dict[str, Any] | None = None) -> torch.Tensor:
        ret = self.decoder(latents) + self.pre_bias
        if self.normalize:
            assert info is not None
            ret = ret * info["std"] + info["mu"]
        return ret

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x, info = self.preprocess(x)
        latents_pre_act = self.encode_pre_act(x)
        latents = self.activation(latents_pre_act)
        recons = self.decode(latents, info)
        return latents_pre_act, latents, recons

    @classmethod
    def from_state_dict(
        cls, state_dict: dict[str, torch.Tensor], strict: bool = True
    ) -> "Autoencoder":
        n_latents, d_model = state_dict["encoder.weight"].shape

        # Retrieve activation
        activation_class_name = state_dict.pop("activation", "ReLU")
        activation_class = ACTIVATIONS_CLASSES.get(activation_class_name, nn.ReLU)
        normalize = activation_class_name == "TopK"
        activation_state_dict = state_dict.pop("activation_state_dict", {})
        if hasattr(activation_class, "from_state_dict"):
            activation = activation_class.from_state_dict(
                activation_state_dict, strict=strict
            )
        else:
            activation = activation_class()
            if hasattr(activation, "load_state_dict"):
                activation.load_state_dict(activation_state_dict, strict=strict)

        autoencoder = cls(n_latents, d_model, activation=activation, normalize=normalize)
        autoencoder.load_state_dict(state_dict, strict=strict)
        return autoencoder

    def state_dict(self, destination=None, prefix="", keep_vars=False):
        sd = super().state_dict(destination, prefix, keep_vars)
        sd[prefix + "activation"] = self.activation.__class__.__name__
        if hasattr(self.activation, "state_dict"):
            sd[prefix + "activation_state_dict"] = self.activation.state_dict()
        return sd


class TiedTranspose(nn.Module):
    def __init__(self, linear: nn.Linear):
        super().__init__()
        self.linear = linear

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert self.linear.bias is None
        return F.linear(x, self.linear.weight.t(), None)

    @property
    def weight(self) -> torch.Tensor:
        return self.linear.weight.t()

    @property
    def bias(self) -> torch.Tensor:
        return self.linear.bias


class TopK(nn.Module):
    def __init__(self, k: int, postact_fn: Callable = nn.ReLU()) -> None:
        super().__init__()
        self.k = k
        self.postact_fn = postact_fn

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        topk = torch.topk(x, k=self.k, dim=-1)
        values = self.postact_fn(topk.values)
        result = torch.zeros_like(x)
        result.scatter_(-1, topk.indices, values)
        return result

    def state_dict(self, destination=None, prefix="", keep_vars=False):
        state_dict = super().state_dict(destination, prefix, keep_vars)
        state_dict.update({prefix + "k": self.k, prefix + "postact_fn": self.postact_fn.__class__.__name__})
        return state_dict

    @classmethod
    def from_state_dict(cls, state_dict: dict[str, torch.Tensor], strict: bool = True) -> "TopK":
        k = state_dict["k"]
        postact_fn = ACTIVATIONS_CLASSES[state_dict["postact_fn"]]()
        return cls(k=k, postact_fn=postact_fn)


ACTIVATIONS_CLASSES = {
    "ReLU": nn.ReLU,
    "Identity": nn.Identity,
    "TopK": TopK,
}


In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# # Import your existing components from the notebook
# from your_notebook import Autoencoder           # base autoencoder class
# from your_notebook import beta_schedule         # dynamic beta schedule function
# from your_notebook import kl_divergence_loss    # KL divergence sparsity loss

# -----------------------------------
# 1) Define Matryoshka Sparse Autoencoder with Batch‐TopK sparsity
# -----------------------------------
class MatryoshkaAutoencoder(Autoencoder):
    def __init__(
        self,
        latent_dim: int,
        input_dim: int,
        group_sizes: list[int],
        k: int,
        threshold_beta: float = 0.999,
        activation: nn.Module = nn.ReLU(),
        tied: bool = True,
        normalize: bool = True
    ):
        """
        Matryoshka SAE with:
          - nested prefixes (group_sizes)
          - a learnable threshold for sparsity
          - global batch‐topK masking (k nonzeros per sample)
        """
        super().__init__(latent_dim, input_dim,
                         activation=activation,
                         tied=tied, normalize=normalize)
        assert group_sizes[-1] == latent_dim, "last prefix must equal latent_dim"
        self.group_sizes = group_sizes
        self.k = k
        self.threshold_beta = threshold_beta
        # threshold < 0 indicates "not yet initialized"
        self.register_buffer('threshold', torch.tensor(-1.0))

    def encode(self, x: torch.Tensor):
        # preprocess + encode pre‐activation
        x_norm, info = self.preprocess(x)
        z_pre = self.encode_pre_act(x_norm)
        z = self.activation(z_pre)  # [B, D]

        # 1) update dynamic threshold based on current activations
        act = z[z > 0]
        if act.numel() > 0:
            min_act = act.min().detach()
            if self.threshold < 0:
                self.threshold = min_act
            else:
                self.threshold = (self.threshold_beta * self.threshold
                                  + (1 - self.threshold_beta) * min_act)

        # 2) apply threshold mask
        z_thresh = z * (z > self.threshold)

        # 3) flatten and take top-(k * B) activations across the batch
        B, D = z_thresh.shape
        flat = z_thresh.view(-1)
        topk = flat.topk(self.k * B, sorted=False)
        mask = torch.zeros_like(flat)
        mask[topk.indices] = topk.values
        f = mask.view(B, D)

        return f, info

    def forward(self, x: torch.Tensor):
        """
        Returns:
          f      = sparse code [B, D]
          recons = list of reconstructions [B, input_dim] for each prefix
        """
        f, info = self.encode(x)
        recons = []
        for m in self.group_sizes:
            f_m = f.clone()
            f_m[:, m:] = 0          # zero out units beyond prefix m
            x_hat = self.decode(f_m, info)
            recons.append(x_hat)
        return f, recons

# -----------------------------------
# 2) Prepare DataLoaders for MF outputs
# -----------------------------------
# Assume you have two numpy arrays or tensors:
#   user_embeddings: shape [N_users, K]
#   item_embeddings: shape [N_items, K]
user_tensor = torch.tensor(mf_recommender.P, dtype=torch.float32)
item_tensor = torch.tensor(mf_recommender.Q, dtype=torch.float32)

user_loader = DataLoader(user_tensor, batch_size=256, shuffle=True)
item_loader = DataLoader(item_tensor, batch_size=256, shuffle=True)

# -----------------------------------
# 3) Train function for Matryoshka SAE
# -----------------------------------
def train_matryoshka(
    sae: MatryoshkaAutoencoder,
    user_loader: DataLoader,
    item_loader: DataLoader,
    num_epochs: int = 30,
    lr: float = 1e-3,
    mse_weight: float = 8.0,
    inner_start: float = 0.0,
    inner_end: float = 20.0,
    warmup: int = 5,
    device: str = 'cuda'
):
    """
    Trains Matryoshka SAE with:
      1) Nested reconstruction losses per prefix
      2) Batch‐TopK sparsity (no L1/KL penalties needed)
      3) Dynamic inner‐product loss on final reconstruction
    """
    # sae.to(device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        epoch_loss = 0.0

        for u, i in zip(user_loader, item_loader):
            # u = u.to(device)
            # i = i.to(device)

            # encode + reconstruct
            f_u, recons_u = sae(u)
            f_i, recons_i = sae(i)

            # 1) nested reconstruction loss (sum over all prefix levels)
            recon_u = sum(F.mse_loss(r, u) for r in recons_u)
            recon_i = sum(F.mse_loss(r, i) for r in recons_i)
            recon_term = mse_weight * (recon_u + recon_i)

            # 2) dynamic inner‐product loss on highest‐level reconstruction
            u_hat = recons_u[-1]
            i_hat = recons_i[-1]
            inner_orig = u @ i.T
            inner_recon = u_hat @ i_hat.T
            beta = beta_schedule(
                epoch - 1, num_epochs,
                beta_start=inner_start,
                beta_end=inner_end,
                warmup=warmup
            )
            inner_term = beta * F.mse_loss(inner_recon, inner_orig)

            # total loss
            loss = recon_term + inner_term

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * u.size(0)

        avg_loss = epoch_loss / len(user_loader.dataset)
        print(f"Epoch {epoch}/{num_epochs} — avg_loss: {avg_loss:.4f}")


NameError: name 'Autoencoder' is not defined

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# # Import your existing components from the notebook
# from your_notebook import Autoencoder           # base autoencoder class
# from your_notebook import beta_schedule         # dynamic beta schedule function
# from your_notebook import kl_divergence_loss    # KL divergence sparsity loss

# -----------------------------------
# 1) Define Matryoshka Sparse Autoencoder
# -----------------------------------
class MatryoshkaAutoencoder(Autoencoder):
    def __init__(self,
                 latent_dim: int,
                 input_dim: int,
                 group_sizes: list[int],
                 activation: nn.Module = nn.ReLU(),
                 tied: bool = True,
                 normalize: bool = True):
        """
        Extends your base Autoencoder to implement Matryoshka-style nested codebooks.

        Args:
            latent_dim: total number of latent units (D)
            input_dim:  dimension of input features (e.g. K from MF embeddings)
            group_sizes: list of prefix lengths [m1, m2, ..., D] that sum to latent_dim
            activation:  nonlinearity to apply after encoder pre-activation
            tied:        whether decoder weights are tied to encoder
            normalize:   whether to apply normalization hooks from base class
        """
        super().__init__(latent_dim, input_dim,
                         activation=activation,
                         tied=tied, normalize=normalize)
        assert group_sizes[-1] == latent_dim, "the last prefix must equal latent_dim"
        self.group_sizes = group_sizes

    def forward(self, x: torch.Tensor):
        # 1) Preprocess (e.g. normalization) and encode
        x_norm, info = self.preprocess(x)
        z_pre  = self.encode_pre_act(x_norm)
        z = self.activation(z_pre)  # apply activation (e.g. ReLU)

        # 2) Multi-level reconstruction: for each prefix length m,
        #    mask off latents beyond m and decode
        recons = []
        for m in self.group_sizes:
            z_masked = z.clone()
            z_masked[:, m:] = 0  # zero out units beyond prefix m
            x_hat = self.decode(z_masked, info)
            recons.append(x_hat)

        # returns:
        #  lat_pre        pre-activation latents [B, D]
        #  lat            post-activation latents [B, D]
        #  reconstructions list of [B, input_dim] for each prefix
        return z_pre, z, recons

# -----------------------------------
# 2) Prepare DataLoaders for MF outputs
# -----------------------------------
# Assume you have two numpy arrays or tensors:
#   user_embeddings: shape [N_users, K]
#   item_embeddings: shape [N_items, K]
user_tensor = torch.tensor(mf_recommender.P, dtype=torch.float32)
item_tensor = torch.tensor(mf_recommender.Q, dtype=torch.float32)

user_loader = DataLoader(interaction_embeddings, batch_size=256, shuffle=True)
item_loader = DataLoader(item_tensor, batch_size=256, shuffle=True)

# -----------------------------------
# 3) Train function for Matryoshka SAE
# -----------------------------------
def train_matryoshka(
    sae: MatryoshkaAutoencoder,
    user_loader: DataLoader,
    item_loader: DataLoader,
    num_epochs: int = 30,
    lr: float = 1e-3,
    mse_weight: float = 8.0,
    l1_weight: float = 0.3,
    kl_weight: float = 0.003,
    inner_start: float = 0.0,
    inner_end: float = 20.0,
    warmup: int = 5,
    sparsity_target: float = 0.1,
    device: str = 'cuda'
):
    """
    trains MatryoshkaSAEs on user and item embeddings.
    """
    # sae.to(device)
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)

    for epoch in range(1, num_epochs+1):
        epoch_loss = 0.0

        # Iterate in parallel over user and item batches
        for u,i in zip(user_loader, item_loader):
            # u = batch_u.to(device)  # user MF embeddings
            # i = batch_i.to(device)  # item MF embeddings

            # Forward pass through autoencoder
            _, z_u, recons_u = sae(u)
            _, z_i, recons_i = sae(i)


            # 1) Nested reconstruction loss (sum over all prefix levels)
            recon_u_loss = sum(F.mse_loss(r, u) for r in recons_u)
            recon_i_loss = sum(F.mse_loss(r, i) for r in recons_i)
            recon_term = mse_weight * (recon_u_loss + recon_i_loss)

            # 2) Sparsity penalties on full latent vectors- z
            l1_term = l1_weight * (z_u.abs().mean() + z_i.abs().mean())
            kl_term = kl_weight * (
                kl_divergence_loss(z_u, sparsity_target) +
                kl_divergence_loss(z_i, sparsity_target)
            )

            # 3) Dynamic inner-product loss on full reconstructions
            full_u_hat = recons_u[-1]  # highest-level reconstruction
            full_i_hat = recons_i[-1]
            inner_orig = u @ i.T
            inner_recons = full_u_hat @ full_i_hat.T

            beta = beta_schedule(
                epoch-1, num_epochs,
                beta_start=inner_start,
                beta_end=inner_end,
                warmup=warmup
            )
            inner_term = beta * F.mse_loss(inner_recons, inner_orig)

            # Total loss
            loss = recon_term + l1_term + kl_term + inner_term

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * u.size(0)

        avg_loss = epoch_loss / len(user_loader.dataset)
        print(f"Epoch {epoch}/{num_epochs} — avg_loss: {avg_loss:.4f}")




In [ ]:
import optuna
import torch

# --- Replace these with your actual imports/functions ---
# from your_module import train_autoencoder_grad_beta, ms_score_new, df_cosine_sim_matrix

def objective(trial):
    # 1) Suggest hyperparameters
    mse_w   = trial.suggest_loguniform('mse_weight', 1e-1, 1e1)    # 0.1 → 10
    l1_w    = trial.suggest_loguniform('l1_weight', 1e-5, 1e0)    # 1e‑5 → 1
    kl_w    = trial.suggest_loguniform('kl_weight', 1e-5, 1e-1)   # 1e‑5 → 0.1
    inner_end = trial.suggest_uniform('beta_end', 0.0, 20.0)       # 0 → 20
    warmup  = trial.suggest_int('warmup', 5, 15)  # 5 → 15 epochs
    # latent_dim = trial.suggest_int('warmup', 70, 90)  # 70 → 90

    # 2) Train your SAE with this config
    K = 80 #user_tensor.shape[1]
    # Define nested prefixes: e.g. 25%, 50%, 100% of K
    prefixes = [K // 4, K // 2, K]
    sae_shared = MatryoshkaAutoencoder(latent_dim=K, input_dim=K, group_sizes=prefixes,k=10)

    train_matryoshka(
        sae_shared,
        user_loader,
        item_loader,
        num_epochs=30,
        lr=1e-3,
        mse_weight = mse_w,
        inner_end= inner_end,
        warmup     = warmup
    )

    # 3) Extract the item latents and compute MS Score
    # _,latents_items,_ = sae_shared(dataset_items)  # assume shape [N_items, latent_dim]
    latents_items,_ = sae_shared(dataset_items)  # assume shape [N_items, latent_dim]
    avg_ms = ms_score_new(df_cosine_sim_matrix, latents_items)

    return float(avg_ms)

# 4) Create and run the study
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=10)

# # 5) Inspect the best trial
# print("Best MS Score:", study.best_value)
# print("Best hyperparameters:")
# for key, val in study.best_params.items():
#     print(f"  {key}: {val}")


# 5a) המר לתוך DataFrame
df_trials = study.trials_dataframe()

# 5b) הצגה על המסך
# print(df_trials)

# 5c) אופציונלי: מיון לפי ביצועים ושמירה ל־CSV
df_sorted = df_trials.sort_values("value", ascending=False)
print("\nTop 5 trials:")
# print(df_sorted.head(5))
# df_sorted.to_csv(OUTPUTS_DIR+'dors/220725_iter_predLoss/optuna_sae_trials.csv', index=False)
df_sorted.to_csv(OUTPUTS_DIR+'dors/220725_iter_predLoss/optuna_matriyoshka_260725_1.csv', index=False)

# 5d) גרף לדוגמה: MS Score מול beta_end
plt.figure(figsize=(6,4))
plt.scatter(df_trials["params_beta_end"], df_trials["value"])
plt.xlabel("beta_end")
plt.ylabel("MS Score")
plt.title("MS Score vs. beta_end")
plt.show()

df_trials

In [ ]:
# 5) Inspect the best trial
print("Best MS Score:", study.best_value)
print("Best hyperparameters:")
for key, val in study.best_params.items():
    print(f"  {key}: {val}")

In [ ]:
# -----------------------------------
# 4) Run training
# -----------------------------------
K = user_tensor.shape[1]
# Define nested prefixes: e.g. 25%, 50%, 100% of K
prefixes = [K // 4, K // 2, K]

sae_shared = MatryoshkaAutoencoder(latent_dim=K, input_dim=K, group_sizes=prefixes, k=10)

train_matryoshka(
    sae_shared,
    user_loader,
    item_loader,
    num_epochs=30,
    lr=1e-3,
    mse_weight= 3.7183641805732095,
    # l1_weight  = 0.008758419327655994,
    # kl_weight  =  0.050771314168206266,
    warmup     = 5,
    inner_end=11.84829137724085,
)
# train_matryoshka(
#     sae_shared,
#     user_loader,
#     item_loader,
#     num_epochs=30,
#     lr=1e-3
# )

In [ ]:
model_name ='SAE_NCF_310725_1650_constInner_2_mse0.11714261837903445_l10.0007907172569000147_kl0.9532963530667937_output_13.589849432397903'

In [ ]:
torch.save(sae_model, Path(OUTPUTS_DIR,f'dors/290725/{model_name}.pth'))

In [ ]:
# Train Matryoshka SAE on same data as SAE
from torch.utils.data import DataLoader

K = dataset_items.shape[1]          # input dim (same as SAE)
latent_dim_mat = K                  # latent dim = K (same as SAE)
prefixes = [K // 4, K // 2, K]     # nested prefix sizes: 25%, 50%, 100%

# Build Matryoshka model (same architecture as SAE)
mat_sae = MatryoshkaAutoencoder(
    latent_dim=latent_dim_mat,
    input_dim=K,
    group_sizes=prefixes,
    activation=torch.nn.ReLU(),
    tied=True,
    normalize=True
)

# DataLoaders using same tensors as SAE training
user_loader_mat = DataLoader(dataset_users, batch_size=256, shuffle=True, drop_last=True)
item_loader_mat = DataLoader(dataset_items, batch_size=256, shuffle=True, drop_last=True)

# Train with best Matryoshka hyperparameters from notebook
train_matryoshka(
    mat_sae,
    user_loader_mat,
    item_loader_mat,
    num_epochs=30,
    lr=1e-3,
    mse_weight=3.7183641805732095,
    warmup=5,
    inner_end=11.84829137724085,
)

print("Matryoshka SAE training complete.")